# 03b. Despliegue del Modelo e Integración RAG Vectorial (Fase de Validación)

## Objetivos de este Notebook
1.  **Carga Dual:** Desplegar el LLM (**Phi-3-mini-4k-instruct**) y el modelo de Embeddings (**BAAI/bge-m3**) en la GPU.
2.  **Conexión Distribuida:** Establecer comunicación con Elasticsearch (alojado en CPU/Valencia) mediante túnel SSH.
3.  **Pipeline RAG Semántico:** Implementar la función de búsqueda vectorial (kNN) con configuración estricta (**Top-k=1**, **Match Perfecto**).
4.  **Validación Técnica:** Realizar las mismas pruebas de cordura que en el BM25 para comparar resultados.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

# 1. Configuración de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware detectado: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

# 2. Carga del Modelo de Embeddings (NUEVO PARA FASE VECTORIAL)
print("Cargando modelo de Embeddings: BAAI/bge-m3...")
embed_model = SentenceTransformer('BAAI/bge-m3', device=device)

# 3. Selección y Carga del LLM 
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
print(f"Cargando modelo LLM: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",             
    torch_dtype=torch.float16    
)

# 4. Crear Pipeline de Generación
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("\n ¡Modelos cargados y listos para inferencia!")

/home/javierruiz/miniconda3/envs/environment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hardware detectado: NVIDIA GeForce RTX 4090
Cargando modelo de Embeddings: BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2908.10it/s, Materializing param=pooler.dense.weight]                               


Cargando modelo LLM: microsoft/Phi-3-mini-4k-instruct...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:01<00:00, 136.59it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the cpu.



 ¡Modelos cargados y listos para inferencia!


In [2]:
# 5. Prueba de Inferencia (Sin RAG todavía)
messages = [
    {"role": "user", "content": "¿Podrías explicarme brevemente qué es la inflación económica?"},
]

generation_args = {
    "max_new_tokens": 150,     
    "return_full_text": False, 
    "temperature": 0.1,        
    "do_sample": True,
}

print(" Generando respuesta...")
output = pipe(messages, **generation_args)
print("\n" + "="*50)
print(output[0]['generated_text'][-1]['content'] if isinstance(output[0]['generated_text'], list) else output[0]['generated_text'])
print("="*50)

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Generando respuesta...

 La inflación económica es un fenómeno que se refiere a la tendencia de los precios de bienes y servicios a aumentar con el tiempo, lo que reduce el poder adquisitivo de la moneda. Esto significa que, con la misma cantidad de dinero, una persona puede comprar menos bienes y servicios de lo que podría haber hecho antes del aumento de precios. La inflación puede ser causada por diversos factores, como un aumento en la oferta de dinero, un aumento en los costos de producción, o una disminución en la demanda de bienes y servicios. A largo plazo, una infla


In [3]:
from elasticsearch import Elasticsearch

# Conexión a Elasticsearch
es = Elasticsearch("http://127.0.0.1:9250")

try:
    if es.ping():
        print("PING EXITOSO")
        print(f" Conectado a: {es.info()['name']}")
    else:
        print("El servidor no responde. ¿Está el túnel abierto?")
except Exception as e:
    print(f"Error inesperado: {e}")

PING EXITOSO
 Conectado a: valencia


In [4]:
def ask_rag_vectorial(query, top_k=1): 
    """
    Función RAG Vectorial: Recupera 1 noticia (kNN) y genera respuesta basada estrictamente en ella.
    Retorna un diccionario con los datos separados.
    """
    # --- A. BÚSQUEDA VECTORIAL (kNN) ---
    vector_pregunta = embed_model.encode(query).tolist()
    
    query_knn = {
        "field": "vector_texto",
        "query_vector": vector_pregunta,
        "k": top_k,
        "num_candidates": 50
    }
    
    try:
        response = es.search(
            index="noticias_tfg_vectores", 
            knn=query_knn, 
            _source=["title", "body", "date", "source"], 
            size=top_k
        )
        hits = response['hits']['hits']
    except Exception as e:
        return {"error": f"Error en Elasticsearch: {e}"}
    
    if not hits:
        return {"error": " NO ENCONTRADO: No hay ninguna noticia cercana a esta búsqueda."}

    # --- B. EXTRACCIÓN ---
    doc = hits[0]['_source']
    contexto_unico = f"""
    TITULO: {doc.get('title')}
    FECHA: {doc.get('date', 'Desconocida')}
    FUENTE: {doc.get('source', 'Desconocida')}
    CONTENIDO: {doc.get('body')}
    """

    # --- C. PROMPT ---
    messages = [
        {"role": "user", "content": f"""
        Eres un sistema de verificación de datos (Fact-Checking). 
        Tu objetivo es responder a la pregunta usando ÚNICAMENTE el texto que te proporciono abajo.

        REGLAS CRÍTICAS:
        1. Si la respuesta no aparece en el texto, responde exactamente: "No tengo información suficiente en mis archivos".
        2. No utilices conocimiento externo.
        3. No menciones otras noticias que no sean la proporcionada.

        ### TEXTO DE REFERENCIA:
        {contexto_unico}
        
        ### PREGUNTA:
        {query}
        """}
    ]

    # --- D. GENERACIÓN ---
    outputs = pipe(
        messages,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0, 
    )
    
    # --- E. RETORNO ESTRUCTURADO ---
    respuesta_limpia = outputs[0]['generated_text'][-1]['content']
    
    return {
        "titulo": doc.get('title'),
        "contenido": doc.get('body'),
        "fecha": doc.get('date'),
        "fuente": doc.get('source'),
        "respuesta_rag": respuesta_limpia
    }

print("Sistema RAG Vectorial (k=1) configurado.")

Sistema RAG Vectorial (k=1) configurado.


In [6]:
pregunta_tfg = "¿Qué aranceles puso Trump a China el 10 de abril de 2025?"

print(f" PREGUNTA: {pregunta_tfg}\n")

print("--- RESPUESTA SLM (SIN RAG) ---")
res_base = pipe([{"role": "user", "content": pregunta_tfg}], max_new_tokens=150)
print(res_base[0]['generated_text'][-1]['content']) 

print("\n")

datos = ask_rag_vectorial(pregunta_tfg)
print("--- RESPUESTA SISTEMA RAG VECTORIAL ---")
if "error" in datos:
    print(datos["error"])
else:
    print(f"NOTICIA: {datos['titulo']}")
    print(f"TEXTO: {datos['contenido'][:300]} (...)")
    print(f"RESPUESTA: {datos['respuesta_rag']}")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 PREGUNTA: ¿Qué aranceles puso Trump a China el 10 de abril de 2025?

--- RESPUESTA SLM (SIN RAG) ---
 Como la fecha mencionada (10 de abril de 2025) es en el futuro con respecto a la fecha de corte del conocimiento de este modelo (2023), no es posible proporcionar detalles precisos sobre los aranceles puestos por el presidente Trump a China en esa fecha. Es importante tener en cuenta que cualquier información sobre eventos futuros no puede ser conocida hasta que sucedan. Los aranceles comerciales son decisiones políticas que pueden cambiar con el tiempo y están sujetas a las políticas económicas y diplomáticas que se desarrollan.




Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RESPUESTA SISTEMA RAG VECTORIAL ---
NOTICIA: Resumen de noticias de aranceles de EE.UU. y Trump, 4 de abril 2025
TEXTO: China dijo que impondrá aranceles recíprocos del 34% a todas las importaciones de Estados Unidos a partir del 10 de abril, en represalia a la guerra comercial del presidente Donald Trump. (...)
RESPUESTA:  No tengo información suficiente en mis archivos.


In [7]:
pregunta_tfg_2 = "¿Qué actor protagoniza la película 'On the Waterfront' (1953) mencionada en el artículo de los Óscar?"

print(f" PREGUNTA: {pregunta_tfg_2}\n")

print("--- RESPUESTA SLM (SIN RAG) ---")
res_base = pipe([{"role": "user", "content": pregunta_tfg_2}], max_new_tokens=150)
print(res_base[0]['generated_text'][-1]['content']) 

print("\n")

datos = ask_rag_vectorial(pregunta_tfg_2)
print("--- RESPUESTA SISTEMA RAG VECTORIAL ---")
if "error" in datos:
    print(datos["error"])
else:
    print(f"NOTICIA: {datos['titulo']}")
    print(f"TEXTO: {datos['contenido'][:300]} (...)")
    print(f"RESPUESTA: {datos['respuesta_rag']}")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 PREGUNTA: ¿Qué actor protagoniza la película 'On the Waterfront' (1953) mencionada en el artículo de los Óscar?

--- RESPUESTA SLM (SIN RAG) ---
 El actor que protagoniza la película 'On the Waterfront' (1953) es Marlon Brando. Esta cinta ganó varios premios Óscar, incluido el de Mejor Actriz para Eva Marie Saint, y fue dirigida por Elia Kazan. Marlon Brando fue nominado para el premio al Mejor Actor por su papel como Terry Malloy.




Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RESPUESTA SISTEMA RAG VECTORIAL ---
NOTICIA: ‘Oppenheimer’, la mejor cinta; gana 7 premios Óscar
TEXTO: El actor Robert Downey Jr se llevó el premio al mejor actor de reparto por su impecable papel en la cinta de Cristopher Nolan. (...)
RESPUESTA:  No tengo información suficiente en mis archivos.
